# Which genes change first after 150 CAGs?

For every phase C gene in striatal neurons (caudate) and cortical pyramidal neurons, this notebook estimates the repeat length at which the gene's expression begins to change, using each nucleus's measured CAG length. It also plots the candidate regulators (MBNL2, CBP, p300, EZH2, HTT and others) against repeat length, nucleus by nucleus.

Data: the processed data deposited with Kashin et al., Cell 2026 (Zenodo 10.5281/zenodo.18202664).

**Before running:** Runtime → Change runtime type → **R**. A High-RAM runtime is safer; the cortical count matrix is about 1.4 GB on disk. Then Runtime → Run all. The whole run takes roughly 20–40 minutes, most of it the download and the cortical fits.

Method, in brief: for each nucleus, a gene's expected count is its total transcript count times that gene's rate in reference nuclei (36–100 CAGs) of the same donor and neuron subtype. A Poisson model then fits a hinge: no change below a threshold length *t*, and a change proportional to length beyond it. The threshold is chosen by scanning *t* from 40 to 300 CAGs; its 95% interval comes from the deviance profile, scaled for overdispersion.

In [ ]:
options(timeout = 7200)
for (p in c("data.table", "Matrix", "ggplot2")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
suppressPackageStartupMessages({library(data.table); library(Matrix); library(ggplot2)})
sessionInfo()$R.version$version.string

## 1. Get the data
Downloads the deposited archive (about 2.9 GB) and extracts only the six files needed. If the download fails, upload `HD_cortex.zip` to the Files panel on the left and run this cell again.

In [ ]:
zip <- "HD_cortex.zip"
url <- "https://zenodo.org/records/18202664/files/HD_cortex.zip?download=1"
if (!file.exists(zip) || file.size(zip) < 2.5e9) {
  status <- system(paste("curl -L --fail -o", zip, shQuote(url)))
  if (status != 0) stop("Download failed. Upload HD_cortex.zip to the Files panel and rerun this cell.")
}
need <- c("HD_cortex/data/deepdive_dge_data/caudate/SPN/raw.dge.rds",
          "HD_cortex/data/deepdive_dge_data/caudate/SPN/summary.rds",
          "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic/raw.dge.rds",
          "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic/summary.rds",
          "HD_cortex/data/gene_expr_models/caudate.stats.txt",
          "HD_cortex/data/gene_expr_models/cortex.stats.txt")
system(paste("unzip -o", zip, paste(need, collapse = " ")))
file.exists(need)

## 2. Load one region at a time
Keeps all nuclei with more than 100 CAGs and a random 4,000 of the reference nuclei (36–100 CAGs), then keeps only the genes needed, so memory stays low.

In [ ]:
CANDIDATES <- c("MBNL1","MBNL2","CREBBP","EP300","EZH2","EED","SUZ12","TCERG1","MED15","HTT","MSH3")
GROUP_COLS <- c("subcluster_name", "subtype", "predClass_allen_gluta_subclass", "predClass_SPN_DI", "cluster_name", "predClass")

prep_region <- function(region, dir, stats_file, p_threshold = 1e-10, n_ref = 4000, seed = 1) {
  raw <- readRDS(file.path(dir, "raw.dge.rds"))
  sd  <- as.data.table(readRDS(file.path(dir, "summary.rds")))
  cat(region, ": summary columns:", paste(head(names(sd), 40), collapse = ", "), "\n")
  grp_col <- intersect(GROUP_COLS, names(sd))[1]
  cat("Neuron subtype column used:", grp_col, "\n")
  sd <- sd[!is.na(CAG_LENGTH) & CAG_LENGTH > 35 & CELL_BARCODE %in% colnames(raw)]
  sd[, GROUP := paste(DONOR, get(grp_col), sep = "|")]
  set.seed(seed)
  ref  <- sd[CAG_LENGTH <= 100]
  if (nrow(ref) > n_ref) ref <- ref[sample(.N, n_ref)]
  cells <- rbind(ref, sd[CAG_LENGTH > 100])
  st <- fread(stats_file)
  st[, SET := fifelse(PHASE_C_PVALUE < p_threshold, fifelse(PHASE_C_EFFECT < 0, "C-", "C+"), "other")]
  genes <- intersect(unique(c(st[SET != "other", GENE], CANDIDATES)), rownames(raw))
  N <- Matrix::colSums(raw[, cells$CELL_BARCODE])
  X <- as.matrix(raw[genes, cells$CELL_BARCODE])
  rm(raw); invisible(gc())
  cat(region, ":", nrow(cells), "nuclei (", sum(cells$CAG_LENGTH > 100), "with >100 CAGs ),", length(genes), "genes\n")
  list(region = region, cells = cells, N = N, X = X, st = st[GENE %in% genes])
}

## 3. Hinge model for each gene

In [ ]:
T_GRID <- seq(40, 300, by = 10)

expected_counts <- function(y, N, grp, is_ref) {
  rate <- tapply(y[is_ref], grp[is_ref], sum) / tapply(N[is_ref], grp[is_ref], sum)
  r <- rate[grp]; r[is.na(r) | r <= 0] <- sum(y[is_ref]) / sum(N[is_ref])
  pmax(as.numeric(N * r), 1e-8)
}

fit_hinge <- function(y, L, mu0) {
  off <- log(mu0)
  dev <- sapply(T_GRID, function(t) {
    h <- pmax(0, L - t) / 100
    f <- tryCatch(glm.fit(cbind(1, h), y, offset = off, family = poisson()), error = function(e) NULL)
    if (is.null(f) || !f$converged) NA_real_ else f$deviance
  })
  if (all(is.na(dev))) return(NULL)
  i <- which.min(dev); t_hat <- T_GRID[i]
  h <- pmax(0, L - t_hat) / 100
  f <- glm.fit(cbind(1, h), y, offset = off, family = poisson())
  phi <- max(1, sum((y - f$fitted.values)^2 / f$fitted.values) / f$df.residual)
  ok <- (dev - dev[i]) / phi <= qchisq(0.95, 1)
  f0 <- glm.fit(matrix(1, length(y), 1), y, offset = off, family = poisson())
  data.table(onset = t_hat, ci_low = min(T_GRID[which(ok)]), ci_high = max(T_GRID[which(ok)]),
             change_per_100 = exp(f$coefficients[2]),
             p_vs_no_change = pchisq((f0$deviance - dev[i]) / phi, 2, lower.tail = FALSE))
}

run_region <- function(R) {
  L <- R$cells$CAG_LENGTH; grp <- R$cells$GROUP; is_ref <- L <= 100
  res <- rbindlist(lapply(rownames(R$X), function(g) {
    y <- R$X[g, ]
    if (sum(y[is_ref]) < 20) return(NULL)
    out <- fit_hinge(y, L, expected_counts(y, R$N, grp, is_ref))
    if (is.null(out)) NULL else cbind(GENE = g, out)
  }))
  res <- merge(res, R$st[, .(GENE, SET, PHASE_C_PVALUE)], by = "GENE")
  res[, region := R$region]
  res
}

## 4. Run both regions
Striatum first (smaller), then cortex. Each region's data are removed from memory before the next is loaded.

In [ ]:
R <- prep_region("striatum", "HD_cortex/data/deepdive_dge_data/caudate/SPN", "HD_cortex/data/gene_expr_models/caudate.stats.txt")
t0 <- Sys.time(); res_str <- run_region(R); print(Sys.time() - t0)
cand_str <- list(cells = R$cells, N = R$N, X = R$X[intersect(CANDIDATES, rownames(R$X)), , drop = FALSE], region = "striatum")
rm(R); invisible(gc())

R <- prep_region("cortex", "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic", "HD_cortex/data/gene_expr_models/cortex.stats.txt")
t0 <- Sys.time(); res_ctx <- run_region(R); print(Sys.time() - t0)
cand_ctx <- list(cells = R$cells, N = R$N, X = R$X[intersect(CANDIDATES, rownames(R$X)), , drop = FALSE], region = "cortex")
rm(R); invisible(gc())

res <- rbind(res_str, res_ctx)
fwrite(res, "phaseC_gene_onsets.csv")
res[SET != "other", .(genes = .N, median_onset = median(onset), onset_below_150 = sum(ci_high < 150), onset_above_200 = sum(ci_low > 200)), by = .(region, SET)]

## 5. When do falling and rising genes start to change?

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
ggplot(res[SET != "other"], aes(onset, fill = SET)) +
  geom_histogram(binwidth = 10, position = "dodge") +
  geom_vline(xintercept = 150, linetype = "dashed") +
  facet_wrap(~ region) +
  scale_fill_manual(values = c("C-" = "#b5651d", "C+" = "#2f6db5"), labels = c("Falling (C-)", "Rising (C+)")) +
  labs(x = "Estimated onset (CAG repeats)", y = "Number of genes", fill = NULL) + theme_minimal(base_size = 14)

## 6. Genes that change in both regions, and the earliest of them

In [ ]:
w <- dcast(res[SET != "other"], GENE ~ region, value.var = c("onset", "ci_low", "ci_high", "SET"))
both <- w[!is.na(onset_striatum) & !is.na(onset_cortex)]
cat(nrow(both), "phase C genes in both regions; correlation of onsets:",
    round(cor(both$onset_striatum, both$onset_cortex, method = "spearman"), 2), "\n")
print(both[order(pmax(onset_striatum, onset_cortex))][1:min(.N, 30)])
fwrite(both, "phaseC_onsets_shared_genes.csv")
for (rg in c("striatum", "cortex")) {
  cat("\nEarliest 20 genes,", rg, "(narrowest intervals first among ties):\n")
  print(res[region == rg & SET != "other"][order(onset, ci_high - ci_low)][1:20, .(GENE, SET, onset, ci_low, ci_high, change_per_100)])
}

## 7. Candidate regulators, nucleus by nucleus
Each point is the mean of observed ÷ expected expression for nuclei in that CAG-length bin, with a 95% bootstrap interval. A value of 1 means no change from the reference nuclei.

In [ ]:
BINS <- c(36, 100, 150, 175, 200, 250, 300, 400, 2000)
cand_table <- function(C) {
  L <- C$cells$CAG_LENGTH; grp <- C$cells$GROUP; is_ref <- L <= 100
  bin <- cut(L, BINS, right = FALSE, dig.lab = 4)
  rbindlist(lapply(rownames(C$X), function(g) {
    y <- C$X[g, ]; r <- y / expected_counts(y, C$N, grp, is_ref)
    rbindlist(lapply(levels(bin), function(b) {
      v <- r[bin == b]; if (length(v) < 10) return(NULL)
      bs <- replicate(500, mean(sample(v, replace = TRUE)))
      data.table(GENE = g, bin = b, n = length(v), mean = mean(v), lo = quantile(bs, .025), hi = quantile(bs, .975))
    }))
  }))[, region := C$region]
}
cand <- rbind(cand_table(cand_str), cand_table(cand_ctx))
cand[, bin := factor(bin, levels = levels(cut(36, BINS, right = FALSE, dig.lab = 4)))]
fwrite(cand, "candidate_regulators_by_CAG_bin.csv")
options(repr.plot.width = 14, repr.plot.height = 9)
ggplot(cand, aes(bin, mean, colour = region, group = region)) +
  geom_hline(yintercept = 1, colour = "grey60") +
  geom_pointrange(aes(ymin = lo, ymax = hi), position = position_dodge(width = 0.4)) +
  geom_line(position = position_dodge(width = 0.4)) +
  facet_wrap(~ GENE, scales = "free_y") +
  scale_colour_manual(values = c(striatum = "#7b5ea7", cortex = "#2f6db5")) +
  labs(x = "CAG repeat length", y = "Expression relative to reference nuclei", colour = NULL) +
  theme_minimal(base_size = 12) + theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 8. Download the results
The three CSV files appear in the Files panel on the left: `phaseC_gene_onsets.csv`, `phaseC_onsets_shared_genes.csv` and `candidate_regulators_by_CAG_bin.csv`. Right-click each to download, or attach them in the chat for interpretation.

## 9. Does a neuron type's own supply of a regulator move its threshold?

If an expanded repeat works by binding up a limited supply of some regulator, a neuron type that makes more of that regulator should start changing at a longer repeat, and one that makes more HTT should start changing at a shorter repeat. This fits one threshold per cortical neuron subtype and compares it with that subtype's baseline level of each candidate, measured in its own nuclei with 36-100 CAGs.

In [ ]:
MIN_HIGH        <- 40     # nuclei above 100 CAGs a subtype needs to be fitted
REF_PER_SUBTYPE <- 1500   # reference nuclei kept per subtype

dir <- "HD_cortex/data/deepdive_dge_data/cortex/glutamatergic"
raw <- readRDS(file.path(dir, "raw.dge.rds"))
sdq <- as.data.table(readRDS(file.path(dir, "summary.rds")))
grp_col <- intersect(c("subcluster_name", "subtype", "predClass_allen_gluta_subclass"), names(sdq))[1]
cat("subtype column used:", grp_col, "\n")
sdq <- sdq[!is.na(CAG_LENGTH) & CAG_LENGTH > 35 & CELL_BARCODE %in% colnames(raw)]
sdq[, SUBTYPE := as.character(get(grp_col))]

st <- fread("HD_cortex/data/gene_expr_models/cortex.stats.txt")
st[, SET := fifelse(PHASE_C_PVALUE < 1e-10, fifelse(PHASE_C_EFFECT < 0, "C-", "C+"), "other")]
cplus  <- intersect(st[SET == "C+", GENE], rownames(raw))
cminus <- intersect(st[SET == "C-", GENE], rownames(raw))
cands  <- intersect(CANDIDATES, rownames(raw))

set.seed(1)
keep <- sdq[, {
  hi <- .SD[CAG_LENGTH > 100]; lo <- .SD[CAG_LENGTH <= 100]
  if (nrow(lo) > REF_PER_SUBTYPE) lo <- lo[sample(.N, REF_PER_SUBTYPE)]
  rbind(hi, lo)
}, by = SUBTYPE]

N  <- Matrix::colSums(raw[, keep$CELL_BARCODE])
Xp <- Matrix::colSums(raw[cplus,  keep$CELL_BARCODE])
Xm <- Matrix::colSums(raw[cminus, keep$CELL_BARCODE])
XC <- as.matrix(raw[cands, keep$CELL_BARCODE])
rm(raw); invisible(gc())
cat(nrow(keep), "nuclei,", length(unique(keep$SUBTYPE)), "subtypes,",
    length(cplus), "rising and", length(cminus), "falling phase C genes\n")

subs <- keep[, .(n = .N, n_high = sum(CAG_LENGTH > 100)), by = SUBTYPE][n_high >= MIN_HIGH]
cat(nrow(subs), "subtypes have at least", MIN_HIGH, "nuclei above 100 CAGs\n")

out <- rbindlist(lapply(subs$SUBTYPE, function(s) {
  i <- which(keep$SUBTYPE == s)
  L <- keep$CAG_LENGTH[i]; is_ref <- L <= 100; grp <- keep$DONOR[i]
  fp <- fit_hinge(Xp[i], L, expected_counts(Xp[i], N[i], grp, is_ref))
  fm <- fit_hinge(Xm[i], L, expected_counts(Xm[i], N[i], grp, is_ref))
  if (is.null(fp) || is.null(fm)) return(NULL)
  base <- rowSums(XC[, i[is_ref], drop = FALSE]) / sum(N[i[is_ref]]) * 1e5
  data.table(SUBTYPE = s, n = length(i), n_high = sum(!is_ref),
             median_UMIs = median(N[i]),
             onset_up = fp$onset, up_lo = fp$ci_low, up_hi = fp$ci_high, up_per_100 = fp$change_per_100,
             onset_dn = fm$onset, dn_lo = fm$ci_low, dn_hi = fm$ci_high, dn_per_100 = fm$change_per_100,
             as.list(base))
}))
fwrite(out, "subtype_thresholds.csv")
print(out[order(onset_up), .(SUBTYPE, n, n_high, onset_up, up_lo, up_hi, onset_dn, dn_lo, dn_hi)])

## 10. Is the threshold related to what the subtype makes?

Spearman correlation across subtypes between each subtype's threshold and its baseline level of each candidate, with the median number of RNA molecules per nucleus included as a check: a subtype that is simply read more deeply will look high on every gene at once.

In [ ]:
vars <- c(intersect(CANDIDATES, names(out)), "median_UMIs")
cors <- rbindlist(lapply(vars, function(g) rbindlist(lapply(c("onset_up", "onset_dn"), function(o) {
  x <- out[[g]]; y <- out[[o]]
  if (length(unique(x)) < 3 || length(unique(y)) < 3) return(NULL)
  ct <- suppressWarnings(cor.test(x, y, method = "spearman"))
  data.table(variable = g, threshold = o, rho = unname(ct$estimate), p = ct$p.value)
}))))
print(cors[order(p)])
cat("\nsubtypes compared:", nrow(out), "- a correlation needs to be strong to show up in so few\n")

options(repr.plot.width = 14, repr.plot.height = 8)
long <- melt(out, id.vars = c("SUBTYPE", "onset_up"), measure.vars = vars,
             variable.name = "gene", value.name = "baseline")
ggplot(long, aes(baseline, onset_up)) +
  geom_point(colour = "#2f6db5", size = 2) +
  geom_smooth(method = "lm", se = FALSE, colour = "#b9b4a8", linewidth = 0.6) +
  facet_wrap(~ gene, scales = "free_x") +
  labs(x = "Baseline expression in that subtype's 36-100 CAG nuclei (per 100,000 RNA molecules)",
       y = "CAG length at which that subtype's rising genes start to change") +
  theme_minimal(base_size = 12)

## 11. Download the results

`subtype_thresholds.csv` holds one row per cortical neuron subtype.